# Stage 1 — Python基礎固め: 医療データの前処理・統計・可視化

このノートは medguide-rag プロジェクトの **Stage 1**（Python でのデータ操作に迷いがない状態を作る）の成果物です。
題材は公開データセット **UCI Heart Disease（Cleveland）**。心臓病の診断データ303件を使って、
pandas / numpy / matplotlib による**データ分析の一連の流れ**を一通り体験します。

> **このノートの読み方（Pythonが分からなくても妥当性を判断できる工夫）**
> このノートの最後に「⑦ 医学的定説との照合」という節があり、そこに**照合スコアカード**（1枚の図）を置いています。
> 分析の**結論**（データが示す傾向）を、医学で一般に正しいとされる**定説**（出典付き・別途調査）と突き合わせ、
> 向きが一致していれば「分析が大きく壊れていない」傍証になり、逆向きなら
> コードを読めなくても「どこかおかしい」と気づけます。**結論から途中経過の妥当性を判断する**ための仕掛けです。

> **注意**: 本ノートは学習・ポートフォリオ用途です。医療上の助言ではありません。
> また、データで見られる相関は定説を「証明」するものではありません（相関≠因果・単一データセットの限界）。

## このノートで学ぶ「データ分析の流れ」

| 段階 | やること | 主に使う道具 |
|---|---|---|
| ① 取得 | データを手元に持ってくる | urllib |
| ② 読み込み | 表としてメモリに載せる | pandas.read_csv |
| ③ 把握 | 形・型・分布をざっと掴む | shape / info / describe |
| ④ 前処理 | 欠損値など「汚れ」を整える | replace / dropna |
| ⑤ 統計 | 代表値で全体像を数値化する | median / mode |
| ⑥ 可視化 | 図にして関係を目で確認する | matplotlib |
| ⑦ 照合 | 結論を医学の定説と突き合わせる | （妥当性チェック・スコアカード） |


## ① データ取得 — まず手元にデータを持つ

分析の第一歩は、対象データを手元のファイルにすることです。ここでは UCI Machine Learning Repository が公開する
Heart Disease データセット（Cleveland, processed）を取得します。

- 出典: UCI ML Repository (ID 45) — https://archive.ics.uci.edu/dataset/45/heart+disease
- ライセンス: CC BY 4.0（帰属表示のもと再配布可）

元データは**列名が付いていない**ため、公式の説明に沿って14列の名前を自分で付けます。
一度 `data/sample/` に CSV として保存しておけば、次回からはネット接続なしで読み込めます（下のコードは
既にファイルがあれば再ダウンロードしません）。


In [ ]:
import io
from pathlib import Path
import urllib.request
import pandas as pd

# ノートは notebooks/ で実行される前提。データは一つ上の data/sample/ に置く
DATA_PATH = Path("../data/sample/heart-disease-cleveland.csv")
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
COLUMNS = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num",
]

if DATA_PATH.exists():
    print(f"ローカルに既存のためダウンロードしません: {DATA_PATH}")
else:
    print("UCI から取得します ...")
    req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=30) as resp:
        raw = resp.read().decode("utf-8")
    df_raw = pd.read_csv(io.StringIO(raw), header=None, names=COLUMNS)
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    df_raw.to_csv(DATA_PATH, index=False, encoding="utf-8")  # cp932環境の罠回避のためUTF-8明示
    print(f"保存しました: {DATA_PATH}（{len(df_raw)}行）")


### 列名の日本語対応表

以降の図や表で使うため、英語の列名と日本語の意味を対応づけておきます。
（データ本体の列名は英語のままにし、表示するときだけ日本語に変換します）


In [ ]:
# 英語列名 → 日本語ラベル（図・表の表示用）
JP = {
    "age": "年齢", "sex": "性別", "cp": "胸痛タイプ", "trestbps": "安静時血圧",
    "chol": "コレステロール", "fbs": "空腹時血糖", "restecg": "安静時心電図",
    "thalach": "最大心拍数", "exang": "運動誘発狭心症", "oldpeak": "ST低下",
    "slope": "ST傾き", "ca": "主要血管数", "thal": "サラセミア",
    "num": "診断(0-4)", "target": "疾患有無",
}
pd.DataFrame({"英語列名": list(JP.keys()), "日本語": list(JP.values())})

## ② 読み込み — 表としてメモリに載せる

保存した CSV を pandas の **DataFrame**（Excel のシートのような2次元の表）として読み込みます。
`head()` は先頭5行を表示するメソッドで、まず「どんな値が入っているか」を目視するのに使います。


In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

## ③ 全体像の把握 — まず形と型を掴む

いきなり細部を見るのではなく、まず**全体の形**を掴みます。

- `shape`: (行数, 列数)
- `info()`: 各列のデータ型と非欠損数。ここで「数値のはずが文字列になっている列」に気づけます
- `describe()`: 数値列の要約統計（件数・平均・標準偏差・最小/最大・四分位）


In [ ]:
print("形 (行, 列):", df.shape)
print()
df.info()

In [ ]:
df.describe()

## ④ 前処理 — 欠損値を確認して整える

このデータでは、欠損値が数値ではなく文字 **`?`** で入っています。`?` が混じっている列は、
pandas が「文字列の列」と誤認するため、`info()` で `object` 型に見えていたはずです。

流れは3手順です。

1. どの列に `?` がいくつあるか数える
2. `?` を欠損（`NaN`）に置き換え、列を数値型に直す
3. 欠損を含む行がごく少数なら**その行を除外**（今回はこの方針。多い場合は補完を検討）


In [ ]:
# 1) '?' がどの列にいくつあるか
mask = df.astype(str) == "?"
missing_by_col = mask.sum()
print("'?' を含む列と件数:")
print(missing_by_col[missing_by_col > 0])

In [ ]:
# 2) '?' を NaN にし、対象列を数値型へ変換
df_clean = df.replace("?", pd.NA).copy()
for col in ["ca", "thal"]:
    df_clean[col] = pd.to_numeric(df_clean[col])

na_by_col = df_clean.isna().sum()
print("NaN 化後の欠損数:")
print(na_by_col[na_by_col > 0])
print()
print("欠損を含む行数:", int(df_clean.isna().any(axis=1).sum()), "/", len(df_clean))

In [ ]:
# 3) 欠損はわずか数行なので、今回は該当行を除外する
df_clean = df_clean.dropna().reset_index(drop=True)
print("除外後の行数:", len(df_clean))
df_clean.head()

## ⑤ 基礎統計 — 代表値で全体像を数値化する

まず目的変数 `num`（心疾患の診断）を扱いやすく **0/1 に二値化**します。元データは 0〜4 の5段階ですが、
0 = 疾患なし、1以上 = 疾患あり、とまとめます。

次に主要な数値列の代表値を出します。ここでは**中央値と最頻値を主役**にします。
平均値は外れ値（極端に大きい/小さい値）に引っ張られやすく、平均だけを見ると全体像を誤解することがあるためです。
中央値（真ん中の値）と最頻値（最も多い値）を併記することで、分布の実態を掴みます。


In [ ]:
# 目的変数を 0/1 に二値化
df_clean["target"] = (df_clean["num"] > 0).astype(int)
print("疾患あり(1) / なし(0) の件数:")
print(df_clean["target"].value_counts().sort_index())

In [ ]:
import numpy as np

num_cols = ["age", "trestbps", "chol", "thalach", "oldpeak"]
stats = pd.DataFrame({
    "中央値": df_clean[num_cols].median(),
    "最頻値": df_clean[num_cols].mode().iloc[0],
    "平均値(参考)": df_clean[num_cols].mean().round(1),
})
stats.index = [JP[c] for c in num_cols]  # 行名を日本語に
stats

## ⑥ 可視化 — 図にして関係を確認し、その場で定説と照合する

数値の表だけでは掴みにくい「分布の形」や「特徴量と疾患の関係」を、matplotlib で図にします。
作った図は `outputs/` に PNG として保存し、後からポートフォリオに載せられるようにします。
図の中の用語（軸・凡例・タイトル）はすべて日本語にします。

**各図の見出しは「結論」を述べる形にし、図の直後に1行の照合メモを置きます。**
その図で分かったこと（データの傾向）を、医学の定説とその場で突き合わせ、向きが一致するか判定します
（全体の要約と出典は最後の⑦スコアカードにまとめます）。

日本語が図の中で文字化けしないよう、日本語対応フォントを指定します。


In [ ]:
import matplotlib
import matplotlib.pyplot as plt

# 日本語フォント（Windows標準）を優先指定。無ければ後続候補にフォールバック
matplotlib.rcParams["font.family"] = ["Yu Gothic", "Meiryo", "MS Gothic", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

OUT = Path("outputs")
OUT.mkdir(exist_ok=True)
print("図の保存先:", OUT.resolve())

### 図1: 疾患あり/なしはほぼ拮抗（学習の偏り懸念は小）

疾患あり/なしの件数バランスを確認します。極端に偏っていると、後のモデル学習で工夫が必要になります。


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
counts = df_clean["target"].value_counts().sort_index()
ax.bar(["疾患なし", "疾患あり"], counts.values, color=["#4c9f70", "#d9534f"])
ax.set_title("図1: 疾患の有無の件数")
ax.set_ylabel("人数")
for i, v in enumerate(counts.values):
    ax.text(i, v + 1, str(int(v)), ha="center")
fig.tight_layout()
fig.savefig(OUT / "01-target-distribution.png", dpi=100)
plt.show()

> **📊 図1の確認**: 疾患なし160 / あり137 でほぼ拮抗（片方が極端に多いと学習が偏るため、まず確認する項目）。
> これは分布バランスの確認で、定説照合の対象ではありません。

### 図2: 相関の当たり付け — 最大心拍数とST低下が疾患と強め

まず全体像として、各数値項目どうし・および「疾患有無」との相関係数を色の濃淡で表します（-1〜+1）。
一番下の「疾患有無」の行を見ると、「疾患と関係の強い項目」の当たりがつきます。
ここで目星をつけた項目（最大心拍数・ST低下・年齢など）を、次から1つずつ図にして定説と照合します。


In [ ]:
corr_cols = ["age", "trestbps", "chol", "thalach", "oldpeak", "target"]
corr = df_clean[corr_cols].corr()
labels = [JP[c] for c in corr_cols]  # 軸ラベルを日本語に

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("図2: 数値項目どうしの相関")
fig.tight_layout()
fig.savefig(OUT / "02-correlation.png", dpi=100)
plt.show()

> **📊 図2の確認**: 疾患有無との相関は 最大心拍数 -0.42・ST低下 +0.42・年齢 +0.23・安静時血圧 +0.15・コレステロール +0.08。
> 最大心拍数とST低下が強め、コレステロールは弱い。この当たりを次から個別に検証します（各項目の定説照合はそれぞれの図で）。

### 図3: 年齢 — 高齢ほど疾患あり（定説と一致 ✓）

年齢のヒストグラムを疾患あり/なしで色分けします。疾患ありの人の年齢が高めに寄っているか、を確認できます。


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(
    [df_clean[df_clean.target == 0]["age"], df_clean[df_clean.target == 1]["age"]],
    bins=15, label=["疾患なし", "疾患あり"], color=["#4c9f70", "#d9534f"],
)
ax.set_title("図3: 年齢の分布（疾患の有無で色分け）")
ax.set_xlabel("年齢")
ax.set_ylabel("人数")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "03-age-hist.png", dpi=100)
plt.show()

> **✅ 照合（年齢）** — データ「疾患ありで高齢：中央値 52→58歳・相関 +0.23」／定説「加齢で疾患リスク上昇」→ **向き一致**

### 図4: 性別 — 男性で疾患あり率が高い（定説と一致 ✓）

性別（1=男性, 0=女性）ごとに「疾患ありの割合」を棒グラフにします。男女でリスクに差があるかを確認します。


In [ ]:
rate = df_clean.groupby("sex")["target"].mean() * 100  # 疾患あり率(%)
n_by_sex = df_clean.groupby("sex")["target"].size()

fig, ax = plt.subplots(figsize=(5, 4))
xlabels = ["女性", "男性"]  # sex=0, 1
ax.bar(xlabels, [rate.get(0, 0), rate.get(1, 0)], color=["#4c9f70", "#d9534f"])
ax.set_title("図4: 性別ごとの疾患あり率")
ax.set_ylabel("疾患あり率 (%)")
for i, s in enumerate([0, 1]):
    ax.text(i, rate.get(s, 0) + 1, f"{rate.get(s, 0):.1f}%\n(n={int(n_by_sex.get(s, 0))})", ha="center")
ax.set_ylim(0, 70)
fig.tight_layout()
fig.savefig(OUT / "04-sex-rate.png", dpi=100)
plt.show()

> **✅ 照合（性別）** — データ「男性で高率：女性 26.0%→男性 55.7%」／定説「中高年は男性が高リスク」→ **向き一致**

### 図5: 最大心拍数 — 疾患ありで低い（定説と一致 ✓）

箱ひげ図で、疾患あり/なしの間で最大心拍数の分布がどう違うかを比べます。
箱が上下にずれていれば、その項目は疾患の判別に効きそう、と読めます。


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
data = [df_clean[df_clean.target == 0]["thalach"], df_clean[df_clean.target == 1]["thalach"]]
ax.boxplot(data, tick_labels=["疾患なし", "疾患あり"])
ax.set_title("図5: 最大心拍数と疾患の関係")
ax.set_ylabel("最大心拍数")
fig.tight_layout()
fig.savefig(OUT / "05-thalach-box.png", dpi=100)
plt.show()

> **✅ 照合（最大心拍数）** — データ「疾患ありで低い：中央値 161→142・相関 −0.42」／定説「運動時に心拍が上がらない（変時性不全）は疾患の予測因子」→ **向き一致**

### 図6: ST低下 — 疾患ありで大きい（定説と一致 ✓）

箱ひげ図で、運動負荷で生じたST低下（心電図の指標）が疾患あり/なしでどう違うかを比べます。


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
data = [df_clean[df_clean.target == 0]["oldpeak"], df_clean[df_clean.target == 1]["oldpeak"]]
ax.boxplot(data, tick_labels=["疾患なし", "疾患あり"])
ax.set_title("図6: 運動によるST低下と疾患の関係")
ax.set_ylabel("ST低下 (oldpeak)")
fig.tight_layout()
fig.savefig(OUT / "06-oldpeak-box.png", dpi=100)
plt.show()

> **✅ 照合（ST低下）** — データ「疾患ありで大きい：中央値 0.2→1.4・相関 +0.42」／定説「運動誘発ST低下は心筋虚血の指標」→ **向き一致**

### 図7: コレステロール — 差は小さいが向きは正（定説と一致 ✓）

箱ひげ図で、血清コレステロールが疾患あり/なしでどう違うかを比べます。
「差が小さい＝おかしい」と早合点しないよう、照合で定説と突き合わせます。


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
data = [df_clean[df_clean.target == 0]["chol"], df_clean[df_clean.target == 1]["chol"]]
ax.boxplot(data, tick_labels=["疾患なし", "疾患あり"])
ax.set_title("図7: コレステロールと疾患の関係")
ax.set_ylabel("コレステロール (mg/dl)")
fig.tight_layout()
fig.savefig(OUT / "07-chol-box.png", dpi=100)
plt.show()

> **✅ 照合（コレステロール）** — データ「差は小さい：中央値 235.5→253.0・相関 +0.08」／定説「総コレステロールの単一測定は相関が弱く出やすい」→ **向き一致（弱いこと自体が定説と符合）**

## ⑦ 医学的定説との照合（結論の妥当性チェック）

ここが本ノートの**核**です。⑥で見た各図の傾向（＝データの結論）を、医学で一般に正しいとされる**定説**
（出典付き・別途調査）と突き合わせ、**向きが一致するか**を確認します。まず下の**スコアカード**で全体を一目で把握できます。

**読み方**: データの傾向と定説の向きが一致していれば、前処理・集計・可視化が大きく壊れていない傍証になります。
逆向きなら、Pythonのコードを読めなくても「データの取り違え・符号ミス・前処理バグ」を疑えます。


In [ ]:
# ⑦の主役: 定説照合スコアカードを1枚の図にまとめる（データ側の数値は df_clean から算出）
from matplotlib.patches import FancyBboxPatch, Rectangle

matplotlib.rcParams["font.family"] = ["Yu Gothic", "Meiryo", "MS Gothic", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

GREEN = "#009E73"   # Okabe-Ito（色覚3型・グレースケール印刷でも判別できる緑）
INK = "#222222"
MUTE = "#555555"

am = df_clean.groupby("target")["age"].median()
tm = df_clean.groupby("target")["thalach"].median()
om = df_clean.groupby("target")["oldpeak"].median()
cm = df_clean.groupby("target")["chol"].median()
sr = df_clean.groupby("sex")["target"].mean() * 100
ct = df_clean[["age", "thalach", "oldpeak", "chol", "target"]].corr()["target"]

rows = [
    ("年齢", f"疾患ありで高い ↑   {am.loc[0]:.0f}→{am.loc[1]:.0f}歳（相関 {ct['age']:+.2f}）",
     "加齢で疾患リスク上昇 ↑", "✓ 一致"),
    ("性別", f"男性で高い ↑   女 {sr.loc[0]:.0f}%→男 {sr.loc[1]:.0f}%",
     "中高年は男性が高リスク ↑", "✓ 一致"),
    ("最大心拍数", f"疾患ありで低い ↓   {tm.loc[0]:.0f}→{tm.loc[1]:.0f}（相関 {ct['thalach']:+.2f}）",
     "上がらないほど疾患あり ↓", "✓ 一致"),
    ("ST低下", f"疾患ありで大きい ↑   {om.loc[0]:.1f}→{om.loc[1]:.1f}（相関 {ct['oldpeak']:+.2f}）",
     "大きいほど心筋虚血 ↑", "✓ 一致"),
    ("コレステロール", f"やや高いが差は小 ↑   {cm.loc[0]:.1f}→{cm.loc[1]:.1f}（相関 {ct['chol']:+.2f}）",
     "単一測定は相関が弱い", "✓ 一致(弱)"),
]

fig, ax = plt.subplots(figsize=(11, 4.8))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

ax.text(0.012, 0.955, "医学的定説との照合スコアカード", fontsize=15, fontweight="bold", color=INK, va="center")
ax.text(0.012, 0.892, "5 項目すべてでデータの向きと医学の定説が一致（5 / 5 ✓）",
        fontsize=11.5, color=GREEN, fontweight="bold", va="center")

X = {"item": 0.015, "data": 0.16, "consensus": 0.585, "judge": 0.845}
header_y = 0.79
ax.text(X["item"], header_y, "項目", fontsize=10.5, fontweight="bold", color=MUTE, va="center")
ax.text(X["data"], header_y, "データが示す向き", fontsize=10.5, fontweight="bold", color=MUTE, va="center")
ax.text(X["consensus"], header_y, "医学の定説", fontsize=10.5, fontweight="bold", color=MUTE, va="center")
ax.text(X["judge"] + 0.0625, header_y, "判定", fontsize=10.5, fontweight="bold", color=MUTE, va="center", ha="center")
ax.plot([0.012, 0.985], [0.745, 0.745], color="#cccccc", lw=1)

row_top = 0.68
row_h = 0.135
for i, (item, data_txt, cons_txt, verdict) in enumerate(rows):
    yc = row_top - i * row_h
    if i % 2 == 1:
        ax.add_patch(Rectangle((0.012, yc - row_h / 2), 0.973, row_h,
                               facecolor="#f5f5f5", edgecolor="none", zorder=0))
    ax.text(X["item"], yc, item, fontsize=11.5, fontweight="bold", color=INK, va="center")
    ax.text(X["data"], yc, data_txt, fontsize=10.5, color=INK, va="center")
    ax.text(X["consensus"], yc, cons_txt, fontsize=10.5, color=INK, va="center")
    ax.add_patch(FancyBboxPatch((X["judge"], yc - 0.043), 0.125, 0.086,
                                boxstyle="round,pad=0.004,rounding_size=0.02",
                                facecolor=GREEN, edgecolor="none", zorder=2))
    ax.text(X["judge"] + 0.0625, yc, verdict, fontsize=10.5, fontweight="bold",
            color="white", ha="center", va="center", zorder=3)

ax.text(0.012, 0.02,
        "※ 向きの一致は「分析が大きく壊れていない」傍証。相関≠因果（証明ではない）。出典は下の表の後に記載。",
        fontsize=8.5, color=MUTE, va="center")

fig.savefig(OUT / "00-scorecard.png", dpi=100, bbox_inches="tight")
plt.show()

> このスコアカードが結論の要約です。判定バッジは **色（青緑）＋✓＋「一致」の文字** を併記しています
> （色が見えにくい環境やグレースケール印刷でも読めるようにするため）。以下は同じ内容を出典付きで詳しく示した表です。

### 総合比較表（出典付きリファレンス）

| 項目 | データが示す向き | 医学の定説 | 判定 |
|---|---|---|---|
| **年齢** | 疾患ありで高齢（52→58歳・+0.23） | 加齢でリスク上昇 | ✅ 一致 |
| **性別** | 男性で高率（女26%→男56%） | 男性が高リスク | ✅ 一致 |
| **最大心拍数** | 疾患ありで低い（161→142・−0.42） | 上がらない＝疾患 | ✅ 一致 |
| **ST低下** | 疾患ありで大きい（0.2→1.4・+0.42） | 大きいほど虚血 | ✅ 一致 |
| **コレステロール** | 差は小さい（235.5→253.0・+0.08） | 単一測定は弱く出る | ✅ 一致(弱) |

**結論**: 5項目すべてで向きが一致。コレステロールの弱い相関も「単一断面の総コレステロールは相関が弱く出やすい」という
定説と符合します。→ 今回の前処理・集計・可視化は**大きくは壊れていない**と判断できます。


### 定説の出典（別途調査・出典付き）

1. **年齢** — 加齢は冠動脈疾患の主要な非修正性危険因子。確度: 高
   - [StatPearls: Risk Factors for Coronary Artery Disease (NCBI)](https://www.ncbi.nlm.nih.gov/books/NBK554410/)
   - [2019 ACC/AHA Guideline on the Primary Prevention of CVD (Circulation)](https://www.ahajournals.org/doi/10.1161/cir.0000000000000678)
2. **性別** — 中高年期は男性のリスクが高い（40歳時点の生涯リスク 男性≈50%/女性≈33%）。確度: 高（差の存在）
   - [Sex Differences in Coronary Heart Disease (Circulation)](https://www.ahajournals.org/doi/10.1161/01.CIR.95.1.252)
   - [Sex Differences in Modifiable Risk Factors and Severity of CAD (JAHA)](https://www.ahajournals.org/doi/10.1161/JAHA.120.017235)
3. **最大心拍数** — 変時性不全（運動時に心拍が上がらない）は冠動脈疾患・予後の独立予測因子。確度: 高
   - [Chronotropic Incompetence: Causes, Consequences, and Management (PMC)](https://pmc.ncbi.nlm.nih.gov/articles/PMC3065291/)
   - [Impaired chronotropic response as a predictor of mortality (PubMed)](https://pubmed.ncbi.nlm.nih.gov/10022108/)
4. **ST低下** — 運動誘発性ST低下（≧1mm）は心筋虚血の所見として広く受け入れられている。確度: 高
   - [ACC/AHA 2002 Guideline Update for Exercise Testing (Circulation)](https://www.ahajournals.org/doi/10.1161/01.cir.0000034670.06526.15)
   - [Exercise Stress Testing: Indications and Common Questions (AAFP)](https://www.aafp.org/afp/2017/0901/p293)
5. **コレステロール** — 高LDLは危険因子だが、総コレステロールの単一測定は変動・推定誤差で相関が弱く出やすい。確度: 高（LDL）/中（単一断面の一般化）
   - [2018 AHA/ACC Guideline on the Management of Blood Cholesterol (Circulation)](https://www.ahajournals.org/doi/10.1161/CIR.0000000000000625)
   - [Measurement of Serum LDL: Possibilities and Limitations (PMC)](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10181074/)

> **重要な但し書き（相関≠因果）**: 上記の定説は母集団レベルの一般的知見です。今回の303件で観測された相関が
> 定説を「証明」するわけではありません（単一データセット・単一時点・相関≠因果の限界）。
> 本節はあくまで「分析結果が常識と大きく矛盾していないか」の**健全性チェック**であり、医療上の結論ではありません。


## まとめ — 分かったことと次への橋渡し

データ分析の基本的な流れ（取得 → 読み込み → 把握 → 前処理 → 統計 → 可視化 → 定説照合）を一通り通しました。

### データから読み取れたこと
1. **件数バランス**（図1）: 疾患あり137 / なし160 でおおむね拮抗
2. **年齢**（図3）: 疾患ありはやや高齢（中央値 52→58歳）
3. **性別**（図4）: 男性の疾患あり率が高い（女26%/男56%）
4. **最大心拍数**（図5）: 疾患と負の相関（−0.42）
5. **ST低下**（図6）: 疾患と正の相関（+0.42）
6. **コレステロール**（図7）: 差は小さい（+0.08）

### 妥当性チェックの結果
上記の傾向は医学の定説と**5項目すべてで向きが一致**しました（詳細は⑦の**スコアカード**・比較表・出典を参照）。
コードの外側（結論と常識の照合）から、分析が大きく壊れていないことを確認できています。

### 次のステージへ
Stage 2 では、前処理・可視化したデータを入力に **scikit-learn で分類モデルを1本完走**させます（学習 → 評価 → 簡易チューニング）。
本ノートで「どの項目が効きそうか」を掴み、かつ「結論を定説で検算する」やり方を確立したことが、
モデルの特徴量選択と結果の妥当性判断に直結します。
